<a href="https://colab.research.google.com/github/sairamsrujan/celebal-excellence-internship/blob/main/Week7_RSaiRamSrujanKumar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Document Question Answering System (RAG)


This notebook builds a Retrieval-Augmented Generation (RAG) system from scratch. The idea is simple: instead of asking a language model to answer from memory, we first pull out the most relevant passages from a document, and then ask the model to answer based on those passages. This way the answer is actually grounded in the document text, not hallucinated.


## Pipeline Overview

There are two phases: indexing (done once) and answering (done per question).

**Indexing**

| Step | What happens | Tool |
|------|-------------|------|
| 1 | Load document into raw text | pypdf / datasets / file upload |
| 2 | Split text into overlapping chunks | sentence-aware splitter |
| 3 | Encode each chunk as a vector | all-MiniLM-L6-v2 (384-d) |
| 4 | Store vectors for similarity search | FAISS (cosine) |

**Answering**

| Step | What happens | Tool |
|------|-------------|------|
| 5 | Encode the question as a vector | all-MiniLM-L6-v2 |
| 6 | Find the most relevant chunks | FAISS + BM25 hybrid + cross-encoder re-ranking |
| 7 | Feed context + question to LLM | Flan-T5 (free) or Claude (if API key is set) |

## 1. Install Dependencies

These are the extra packages needed on top of what Colab already has.

In [ ]:
!pip -q install --upgrade sentence-transformers faiss-cpu rank-bm25 pypdf datasets anthropic
print("Dependencies installed.")

Dependencies installed.


## 2. Imports and Config

All the tunable parameters in one place — chunk size, model names, how many results to retrieve, etc. Makes it easy to experiment later.

In [ ]:
import os, re, json, random, warnings
import numpy as np

warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
random.seed(42)
np.random.seed(42)

CONFIG = {
    # Chunking
    "CHUNK_SIZE": 700,        # characters per chunk
    "CHUNK_OVERLAP": 120,     # overlap between consecutive chunks
    # Embeddings
    "EMBEDDING_MODEL": "sentence-transformers/all-MiniLM-L6-v2",
    # Retrieval
    "TOP_K": 4,               # chunks sent to the LLM
    "CANDIDATE_K": 12,        # candidates fetched before re-ranking
    "USE_HYBRID": True,       # combine BM25 + dense search
    "USE_RERANK": True,       # cross-encoder re-ranking
    "RERANK_MODEL": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    # Generation
    "CLAUDE_MODEL": "claude-sonnet-4-20250514",
    "FREE_MODEL": "google/flan-t5-base",
    "MAX_ANSWER_TOKENS": 1024,
}

try:
    IN_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IN_COLAB = False

print("Running in Colab:", IN_COLAB)
print(json.dumps(CONFIG, indent=2))

Running in Colab: True
{
  "CHUNK_SIZE": 700,
  "CHUNK_OVERLAP": 120,
  "EMBEDDING_MODEL": "sentence-transformers/all-MiniLM-L6-v2",
  "TOP_K": 4,
  "CANDIDATE_K": 12,
  "USE_HYBRID": true,
  "USE_RERANK": true,
  "RERANK_MODEL": "cross-encoder/ms-marco-MiniLM-L-6-v2",
  "CLAUDE_MODEL": "claude-sonnet-4-20250514",
  "FREE_MODEL": "google/flan-t5-base",
  "MAX_ANSWER_TOKENS": 1024
}


## 3. Document Ingestion

The first step is turning documents into plain text. This handles PDFs (via pypdf), raw text files, Hugging Face datasets, and Colab file uploads. There's also a built-in sample document (a fictional solar cooperative handbook) so you can test the whole pipeline without uploading anything.

In [ ]:
# Step 3a: the built-in sample document
SAMPLE_DOCUMENT = '''GreenLeaf Community Solar Cooperative - Member Handbook (2024 Edition)

1. About GreenLeaf
GreenLeaf Community Solar Cooperative is a member-owned organisation that lets
households share the benefits of clean solar power without installing panels on
their own roofs. GreenLeaf was founded in 2016 by Maya Okonkwo, a former
electrical engineer who wanted to make renewable energy affordable for renters
and apartment dwellers. The cooperative is guided by three principles: shared
ownership, transparent billing, and community reinvestment. Each year, 15
percent of the cooperative net profits are reinvested into local community
projects such as school energy-education programmes, tree planting, and grants
for low-income households to improve home insulation.

2. Our Solar Arrays
GreenLeaf operates two solar arrays. The primary site is the Ridgeway array,
located on twelve acres of reclaimed farmland outside the town of Ridgeway. The
Ridgeway array contains 4,200 solar panels and has a peak generating capacity of
about 1.6 megawatts. The second, smaller site is the Bellhaven array, which
contains 2,800 panels and mostly serves members in the eastern districts.
Together the two arrays generate enough electricity in a typical year to power
roughly 900 homes. All electricity produced is fed into the regional grid, and
members receive credits based on their share of production.

3. Membership Tiers and Fees
GreenLeaf offers three membership tiers. The Basic tier costs 60 dollars per
year and entitles a member to a small fixed share of production suitable for a
studio or one-bedroom home. The Standard tier costs 120 dollars per year and is
the most popular option; it includes a larger production share, quarterly usage
reports, and access to the member mobile app. The Premium tier costs 240 dollars
per year, includes the largest production share, priority customer support, and
an annual in-person energy review. Members may upgrade or downgrade their tier
once per calendar year at no extra charge.

4. How Energy Credits Work
Every month, GreenLeaf measures the total number of kilowatt-hours generated by
both arrays. Each member is allocated a production share proportional to their
membership tier. A member monthly energy credit is calculated by multiplying
their production share, expressed in kilowatt-hours, by the local utility retail
rate for electricity. For example, if a Standard member share for the month is
90 kilowatt-hours and the retail rate is 18 cents per kilowatt-hour, the member
receives a credit of 16 dollars and 20 cents on their utility bill. Credits that
exceed a member bill in a given month roll over to the following month.

5. Payments and Billing
Membership fees are billed annually on the anniversary of the join date. Energy
credits appear automatically on each member regular electricity bill through an
arrangement with the local utility, so members do not receive a separate invoice
from GreenLeaf for credits. For any billing questions, members should email the
support team at support@greenleaf.coop or call the member line during business
hours. The cooperative aims to answer all billing enquiries within two business
days.

6. Leaving the Cooperative
Members may leave GreenLeaf at any time by giving 30 days written notice. If a
member chooses to leave within the first year of joining, an early-termination
fee of 75 dollars applies, which covers the administrative cost of onboarding.
After the first full year, there is no early-termination fee, and any remaining
credit balance is paid out to the member within 60 days of departure. Membership
is not transferable between households, but a member who moves within the
service area may keep their membership at the new address.

7. Maintenance and Safety
The arrays are inspected quarterly by certified technicians. Panels are cleaned
twice a year to keep generation efficient, and the inverters are tested for
faults during every inspection. Members are never expected to perform any
maintenance themselves. If a member notices a service outage or a billing
anomaly, they should report it through the member app or by emailing the support
team. GreenLeaf maintains insurance covering both arrays against storm and fire
damage.

8. Governance
GreenLeaf is governed by a nine-person board of directors elected by the
membership. Every member, regardless of tier, has one vote in board elections
under the principle of one member, one vote. The board meets quarterly, and an
annual general meeting is held each spring where members review the financial
report, vote on major decisions, and propose new community projects. Minutes of
every board meeting are published on the member portal within two weeks.

9. Contact and Further Information
The cooperative head office is located in the Ridgeway community centre. General
enquiries can be sent to hello@greenleaf.coop, while billing and account
questions should go to support@greenleaf.coop. New members can join online
through the GreenLeaf website, and the full terms of membership are available in
the member portal. GreenLeaf thanks every member for supporting locally owned,
community-driven renewable energy.'''

In [ ]:
def load_text_file(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def load_pdf(path):
    from pypdf import PdfReader
    reader = PdfReader(path)
    return "\n\n".join((page.extract_text() or "") for page in reader.pages)

def load_hf_dataset(name="vectara/open_ragbench", split="train", text_field=None, max_docs=20):
    from datasets import load_dataset
    ds = load_dataset(name, split=split)
    if text_field is None:
        for k, v in ds[0].items():
            if isinstance(v, str) and len(v) > 40:
                text_field = k
                break
        text_field = text_field or ds.column_names[0]
    n = min(max_docs, len(ds))
    return "\n\n".join(str(ds[i][text_field]) for i in range(n))

def colab_upload():
    from google.colab import files
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    if fname.lower().endswith(".pdf"):
        with open(fname, "wb") as f:
            f.write(uploaded[fname])
        return load_pdf(fname), fname
    return uploaded[fname].decode("utf-8", errors="ignore"), fname

def ingest(source="sample", path=None, **kwargs):
    """Load a document from various sources and return (text, metadata)."""
    if source == "sample":
        return SAMPLE_DOCUMENT.strip(), {"source": "built-in sample document"}
    if source == "text":
        return load_text_file(path), {"source": "text file: " + str(path)}
    if source == "pdf":
        return load_pdf(path), {"source": "pdf: " + str(path)}
    if source == "hf":
        return load_hf_dataset(**kwargs), {"source": "huggingface: " + kwargs.get("name", "vectara/open_ragbench")}
    if source == "upload":
        text, fname = colab_upload()
        return text, {"source": "uploaded: " + fname}
    raise ValueError("Unknown source: " + str(source))

# Default: use the built-in sample
raw_text, doc_meta = ingest("sample")
print("Source     :", doc_meta["source"])
print("Characters :", len(raw_text))
print()
print("Preview:")
print(raw_text[:600], "...")

Source     : built-in sample document
Characters : 5180

Preview:
GreenLeaf Community Solar Cooperative - Member Handbook (2024 Edition)

1. About GreenLeaf
GreenLeaf Community Solar Cooperative is a member-owned organisation that lets
households share the benefits of clean solar power without installing panels on
their own roofs. GreenLeaf was founded in 2016 by Maya Okonkwo, a former
electrical engineer who wanted to make renewable energy affordable for renters
and apartment dwellers. The cooperative is guided by three principles: shared
ownership, transparent billing, and community reinvestment. Each year, 15
percent of the cooperative net profits are rei ...


## 4. Text Chunking

We can't just throw the entire document at the retriever — it needs to be split into smaller passages. I'm using ~700 character chunks with 120 characters of overlap. The overlap is done at sentence boundaries so that facts sitting at the edge of a chunk still show up in the neighboring chunk too.

Chunk size is a tradeoff: too small and you lose context, too big and the retrieval gets diluted. We'll experiment with different sizes later in Section 11.

In [ ]:
def chunk_text(text, chunk_size=None, overlap=None):
    chunk_size = chunk_size or CONFIG["CHUNK_SIZE"]
    overlap = overlap or CONFIG["CHUNK_OVERLAP"]

    # Split into paragraphs and normalize whitespace
    paragraphs = [re.sub(r"\s+", " ", p).strip()
                  for p in re.split(r"\n\s*\n", text) if p.strip()]

    # Break paragraphs into sentences
    sentences = []
    for p in paragraphs:
        sentences.extend(s.strip() for s in re.split(r"(?<=[.!?])\s+", p) if s.strip())

    chunks, cur, cur_len = [], [], 0
    for sent in sentences:
        if cur and cur_len + len(sent) + 1 > chunk_size:
            chunks.append(" ".join(cur))
            # Keep trailing sentences as overlap for the next chunk
            keep, klen = [], 0
            for prev in reversed(cur):
                if keep and klen + len(prev) + 1 > overlap:
                    break
                keep.insert(0, prev)
                klen += len(prev) + 1
            cur, cur_len = keep, klen
        cur.append(sent)
        cur_len += len(sent) + 1
    if cur:
        chunks.append(" ".join(cur))

    return [{"id": i, "text": c, "n_chars": len(c)} for i, c in enumerate(chunks)]

chunks = chunk_text(raw_text)
avg = sum(c["n_chars"] for c in chunks) / max(1, len(chunks))
print("Chunks created :", len(chunks))
print("Chunk size     :", CONFIG["CHUNK_SIZE"], "chars, overlap", CONFIG["CHUNK_OVERLAP"], "chars")
print("Avg chunk size : {:.0f} chars".format(avg))
print()
print("Example chunk:")
print(chunks[1]["text"] if len(chunks) > 1 else chunks[0]["text"])

Chunks created : 10
Chunk size     : 700 chars, overlap 120 chars
Avg chunk size : 613 chars

Example chunk:
The cooperative is guided by three principles: shared ownership, transparent billing, and community reinvestment. Each year, 15 percent of the cooperative net profits are reinvested into local community projects such as school energy-education programmes, tree planting, and grants for low-income households to improve home insulation. 2. Our Solar Arrays GreenLeaf operates two solar arrays. The primary site is the Ridgeway array, located on twelve acres of reclaimed farmland outside the town of Ridgeway. The Ridgeway array contains 4,200 solar panels and has a peak generating capacity of about 1.6 megawatts.


## 5. Embeddings

Each chunk gets converted to a 384-dimensional vector using `all-MiniLM-L6-v2`. It's a small model (~22M parameters) that runs fine on CPU but still does well on semantic similarity benchmarks. The vectors are L2-normalized so dot product = cosine similarity.

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model:", CONFIG["EMBEDDING_MODEL"])
embedder = SentenceTransformer(CONFIG["EMBEDDING_MODEL"])
EMBED_DIM = embedder.get_sentence_embedding_dimension()
print("Embedding dimension:", EMBED_DIM)

def embed(texts, batch_size=64):
    return embedder.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")

chunk_embeddings = embed([c["text"] for c in chunks])
print("Chunk embedding matrix:", chunk_embeddings.shape)

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimension: 384
Chunk embedding matrix: (10, 384)


## 6. Vector Store (FAISS)

The vectors go into a FAISS `IndexFlatIP` index. Since the vectors are already normalized, inner product search gives us cosine similarity. This is brute-force exact search, which is fine for a small corpus. For millions of vectors you'd switch to an approximate index like IVF or HNSW.

In [ ]:
import faiss

class VectorStore:
    def __init__(self, dim):
        self.index = faiss.IndexFlatIP(dim)  # inner product on normalized vecs = cosine
        self.chunks = []

    def add(self, embeddings, chunk_list):
        self.index.add(embeddings)
        self.chunks.extend(chunk_list)

    def search(self, query_vec, k=4):
        scores, idxs = self.index.search(query_vec, k)
        results = []
        for score, idx in zip(scores[0], idxs[0]):
            if idx == -1:
                continue
            results.append({**self.chunks[idx], "score": float(score)})
        return results

store = VectorStore(EMBED_DIM)
store.add(chunk_embeddings, chunks)
print("Vector store ready:", store.index.ntotal, "vectors (FAISS IndexFlatIP, cosine).")

Vector store ready: 10 vectors (FAISS IndexFlatIP, cosine).


## 7. Query Processing and Retrieval

The question goes through the same embedding model as the chunks, putting it in the same vector space. FAISS then returns the closest chunks by cosine similarity.

In [ ]:
def embed_query(question):
    return embed([question])

def retrieve(question, k=None):
    k = k or CONFIG["TOP_K"]
    return store.search(embed_query(question), k)

def snippet(text, n=96):
    return " ".join(text.split())[:n]

# Quick test
demo_q = "What is the annual membership fee for a Standard member?"
print("Dense retrieval for:", demo_q)
print()
for rank, r in enumerate(retrieve(demo_q, k=3), 1):
    print("  {}. chunk {} (cosine {:.3f})  {}...".format(rank, r["id"], r["score"], snippet(r["text"])))

Dense retrieval for: What is the annual membership fee for a Standard member?

  1. chunk 3 (cosine 0.425)  The Basic tier costs 60 dollars per year and entitles a member to a small fixed share of product...
  2. chunk 6 (cosine 0.415)  6. Leaving the Cooperative Members may leave GreenLeaf at any time by giving 30 days written not...
  3. chunk 5 (cosine 0.393)  Credits that exceed a member bill in a given month roll over to the following month. 5. Payments...


## 8. Retrieval Optimizations

Dense search is good at understanding meaning but can miss exact keywords (like email addresses, specific numbers, or proper nouns). To fix this I added two things:

1. **Hybrid search** — BM25 scores chunks by keyword overlap, and the results are combined with the dense results using Reciprocal Rank Fusion (RRF).
2. **Cross-encoder re-ranking** — A cross-encoder reads each (question, chunk) pair and re-scores them. This is more accurate than bi-encoder similarity but slower, so we only run it on the shortlisted candidates.

`smart_retrieve()` chains these together: dense → hybrid fusion → re-rank.

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# Tokenizer for BM25
def _tok(text):
    return re.findall(r"\w+", text.lower())

bm25 = BM25Okapi([_tok(c["text"]) for c in chunks])

def bm25_retrieve(question, k=None):
    k = k or CONFIG["TOP_K"]
    scores = bm25.get_scores(_tok(question))
    top = np.argsort(scores)[::-1][:k]
    return [{**chunks[i], "score": float(scores[i])} for i in top]

def rrf_fuse(*rankings, k_rrf=60):
    """Reciprocal Rank Fusion — merges multiple ranked lists without needing tuned weights."""
    scores = {}
    by_id = {}
    for ranking in rankings:
        for rank, item in enumerate(ranking):
            cid = item["id"]
            scores[cid] = scores.get(cid, 0) + 1.0 / (k_rrf + rank + 1)
            by_id[cid] = item
    fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [{**by_id[cid], "score": sc} for cid, sc in fused]

_reranker = None
def get_reranker():
    global _reranker
    if _reranker is None:
        print("Loading re-ranker:", CONFIG["RERANK_MODEL"])
        _reranker = CrossEncoder(CONFIG["RERANK_MODEL"])
    return _reranker

def rerank(question, candidates, k=None):
    k = k or CONFIG["TOP_K"]
    ce = get_reranker()
    logits = ce.predict([(question, c["text"]) for c in candidates])
    ranked = sorted(zip(candidates, logits), key=lambda x: x[1], reverse=True)
    out = []
    for c, logit in ranked[:k]:
        rel = float(1.0 / (1.0 + np.exp(-float(logit))))  # sigmoid for readability
        out.append({**c, "score": rel})
    return out

def smart_retrieve(question, k=None, use_hybrid=None, use_rerank=None):
    """Full retrieval pipeline: dense -> hybrid fusion -> re-rank."""
    k = k or CONFIG["TOP_K"]
    use_hybrid = CONFIG["USE_HYBRID"] if use_hybrid is None else use_hybrid
    use_rerank = CONFIG["USE_RERANK"] if use_rerank is None else use_rerank
    cand_k = CONFIG["CANDIDATE_K"] if (use_hybrid or use_rerank) else k

    dense = retrieve(question, k=cand_k)
    candidates = dense
    if use_hybrid:
        sparse = bm25_retrieve(question, k=cand_k)
        candidates = rrf_fuse(dense, sparse)[:cand_k]
    if use_rerank:
        return rerank(question, candidates, k=k)
    return candidates[:k]

print("Hybrid + re-rank retrieval for:", demo_q)
print()
for rank, r in enumerate(smart_retrieve(demo_q), 1):
    print("  {}. chunk {} (relevance {:.2f})  {}...".format(rank, r["id"], r["score"], snippet(r["text"])))

Hybrid + re-rank retrieval for: What is the annual membership fee for a Standard member?

Loading re-ranker: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

  1. chunk 3 (relevance 0.67)  The Basic tier costs 60 dollars per year and entitles a member to a small fixed share of product...
  2. chunk 5 (relevance 0.05)  Credits that exceed a member bill in a given month roll over to the following month. 5. Payments...
  3. chunk 4 (relevance 0.03)  4. How Energy Credits Work Every month, GreenLeaf measures the total number of kilowatt-hours ge...
  4. chunk 2 (relevance 0.01)  The Ridgeway array contains 4,200 solar panels and has a peak generating capacity of about 1.6 m...


## 9. Answer Generation

The retrieved passages and the question are assembled into a prompt that tells the model to answer strictly from the provided context. If you set an `ANTHROPIC_API_KEY` in Colab secrets, it uses Claude. Otherwise it falls back to Flan-T5 (free, no key needed). Flan-T5 is extractive — it pulls an answer span from the context. Claude will synthesize across passages and add citations.

In [ ]:
import anthropic

def get_anthropic_key():
    """Check Colab secrets and env vars for an API key. Never prompts interactively."""
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        key = None
    if not key:
        key = os.environ.get("ANTHROPIC_API_KEY")
    return key

ANTHROPIC_API_KEY = get_anthropic_key()
USE_CLAUDE = bool(ANTHROPIC_API_KEY)
print("Generation backend:",
      ("Claude (" + CONFIG["CLAUDE_MODEL"] + ")") if USE_CLAUDE
      else ("free model (" + CONFIG["FREE_MODEL"] + ")"))

SYSTEM_PROMPT = (
    "You are a precise question-answering assistant. Answer the user question "
    "using ONLY the provided context passages. If the answer is not contained in "
    "the context, reply exactly: I do not know based on the provided document. "
    "Be concise and cite the passage numbers you used, for example [1], [2]."
)

def build_prompt(question, retrieved):
    context = "\n\n".join(
        "[{}] (chunk {}) {}".format(i + 1, r["id"], r["text"])
        for i, r in enumerate(retrieved)
    )
    return ("Context passages:\n" + context +
            "\n\nQuestion: " + question +
            "\n\nAnswer using only the context above and cite passage numbers.")

_claude_client = None
def generate_with_claude(question, retrieved):
    global _claude_client
    if _claude_client is None:
        _claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    msg = _claude_client.messages.create(
        model=CONFIG["CLAUDE_MODEL"],
        max_tokens=CONFIG["MAX_ANSWER_TOKENS"],
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": build_prompt(question, retrieved)}],
    )
    return "".join(b.text for b in msg.content if b.type == "text").strip()

_free_tok = None
_free_model = None
def generate_with_free_model(question, retrieved):
    global _free_tok, _free_model
    if _free_model is None:
        import torch
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        from transformers.utils import logging as hf_logging
        hf_logging.set_verbosity_error()
        print("Loading free model:", CONFIG["FREE_MODEL"])
        _free_tok = AutoTokenizer.from_pretrained(CONFIG["FREE_MODEL"])
        _free_model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["FREE_MODEL"])
        if torch.cuda.is_available():
            _free_model = _free_model.to("cuda")
    top = retrieved[:3]
    context = "\n".join("[{}] {}".format(i + 1, r["text"][:400]) for i, r in enumerate(top))
    prompt = ("Answer the question using only the context. If the answer is not "
              "present, say: I do not know based on the provided document.\n\n"
              "Context:\n" + context + "\n\nQuestion: " + question + "\nAnswer:")
    inputs = _free_tok(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(_free_model.device) for k, v in inputs.items()}
    output_ids = _free_model.generate(**inputs, max_new_tokens=200)
    return _free_tok.decode(output_ids[0], skip_special_tokens=True).strip()

def generate(question, retrieved):
    if USE_CLAUDE:
        try:
            return generate_with_claude(question, retrieved), "claude:" + CONFIG["CLAUDE_MODEL"]
        except Exception as e:
            print("Claude call failed, falling back to free model:", e)
    return generate_with_free_model(question, retrieved), "free:" + CONFIG["FREE_MODEL"]

Generation backend: free model (google/flan-t5-base)


## 10. End-to-End RAG Pipeline

`rag_answer()` ties everything together: retrieve → fuse → re-rank → generate. It also prints the source passages so you can verify the answer is actually grounded in the document.

In [ ]:
def rag_answer(question, k=None, use_hybrid=None, use_rerank=None, show_sources=True):
    retrieved = smart_retrieve(question, k=k, use_hybrid=use_hybrid, use_rerank=use_rerank)
    answer, backend = generate(question, retrieved)
    print("Q:", question)
    print()
    print("A:", answer)
    print()
    print("backend:", backend)
    if show_sources:
        print("sources:")
        for rank, r in enumerate(retrieved, 1):
            print("  {}. chunk {} (relevance {:.2f})  {}...".format(rank, r["id"], r["score"], snippet(r["text"])))
    print("-" * 80)
    return answer, retrieved

# Demo
_ = rag_answer("What is the annual membership fee for a Standard member, and what does it include?")

Loading free model: google/flan-t5-base


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Q: What is the annual membership fee for a Standard member, and what does it include?

A: 120 dollars

backend: free:google/flan-t5-base
sources:
  1. chunk 3 (relevance 0.56)  The Basic tier costs 60 dollars per year and entitles a member to a small fixed share of product...
  2. chunk 5 (relevance 0.03)  Credits that exceed a member bill in a given month roll over to the following month. 5. Payments...
  3. chunk 4 (relevance 0.01)  4. How Energy Credits Work Every month, GreenLeaf measures the total number of kilowatt-hours ge...
  4. chunk 2 (relevance 0.01)  The Ridgeway array contains 4,200 solar panels and has a peak generating capacity of about 1.6 m...
--------------------------------------------------------------------------------


## 11. Chunk Size Experiment

Does chunk size actually matter? Let's rebuild the index at three different sizes and measure retrieval hit rate (does the right chunk appear in the top-4 results?).

In [ ]:
def build_index_for(chunk_size, overlap):
    ch = chunk_text(raw_text, chunk_size=chunk_size, overlap=overlap)
    emb = embed([c["text"] for c in ch])
    vs = VectorStore(EMBED_DIM)
    vs.add(emb, ch)
    return vs, ch

def retrieval_hit_rate(vs, questions_keywords, k=4):
    hits = 0
    for q, kws in questions_keywords:
        res = vs.search(embed([q]), k)
        joined = " ".join(r["text"].lower() for r in res)
        hits += any(kw.lower() in joined for kw in kws)
    return hits / len(questions_keywords)

EVAL_QUESTIONS = [
    ("What is the annual membership fee for a Standard member?", ["120", "standard"]),
    ("Who founded GreenLeaf and in what year?", ["maya", "2016"]),
    ("How are energy credits calculated each month?", ["kilowatt", "credit"]),
    ("What is the early-termination fee within the first year?", ["75", "termination"]),
    ("How many solar panels are in the Ridgeway array?", ["4,200", "ridgeway"]),
    ("Who do members contact for billing questions?", ["billing", "support@greenleaf"]),
    ("What percentage of profits are reinvested into community projects?", ["15", "percent"]),
]

print("Chunk-boundary experiment (retrieval hit@4):")
print()
for cs, ov in [(400, 60), (700, 120), (1000, 150)]:
    vs, ch = build_index_for(cs, ov)
    hr = retrieval_hit_rate(vs, EVAL_QUESTIONS, k=4)
    print("chunk_size={:>4}  overlap={:>3}  ->  {:>3} chunks | hit@4 = {:.2f}".format(cs, ov, len(ch), hr))

Chunk-boundary experiment (retrieval hit@4):

chunk_size= 400  overlap= 60  ->   25 chunks | hit@4 = 1.00
chunk_size= 700  overlap=120  ->   10 chunks | hit@4 = 1.00
chunk_size=1000  overlap=150  ->    7 chunks | hit@4 = 1.00


## 12. Validation Logs

Running a fixed set of questions through all three retrieval configs (dense-only, hybrid, hybrid+rerank) and checking whether the expected keywords show up in the results. This is a simple but repeatable way to measure retrieval quality.

In [ ]:
def run_validation(questions_keywords):
    configs = [
        ("dense-only",    dict(use_hybrid=False, use_rerank=False)),
        ("hybrid",        dict(use_hybrid=True,  use_rerank=False)),
        ("hybrid+rerank", dict(use_hybrid=True,  use_rerank=True)),
    ]
    summary = {}
    for name, opts in configs:
        hits = 0
        print()
        print("Retrieval config:", name)
        for q, kws in questions_keywords:
            res = smart_retrieve(q, k=CONFIG["TOP_K"], **opts)
            joined = " ".join(r["text"].lower() for r in res)
            hit = any(kw.lower() in joined for kw in kws)
            hits += hit
            ids = [r["id"] for r in res]
            mark = "PASS" if hit else "MISS"
            print("  [{}] top_chunks={} expects~{} | {}".format(mark, ids, kws, q[:52]))
        summary[name] = hits / len(questions_keywords)
    print()
    print("Retrieval hit@{} summary:".format(CONFIG["TOP_K"]))
    for name, hr in summary.items():
        print("  {:<15}: {:.2f}".format(name, hr))
    return summary

RETRIEVAL_SUMMARY = run_validation(EVAL_QUESTIONS)

print()
print("Generated answers (best config):")
print()
GENERATED = []
for q, _kws in EVAL_QUESTIONS:
    ans, _ = rag_answer(q, show_sources=False)
    GENERATED.append((q, ans))


Retrieval config: dense-only
  [PASS] top_chunks=[3, 6, 5, 4] expects~['120', 'standard'] | What is the annual membership fee for a Standard mem
  [PASS] top_chunks=[0, 8, 9, 6] expects~['maya', '2016'] | Who founded GreenLeaf and in what year?
  [PASS] top_chunks=[4, 5, 3, 2] expects~['kilowatt', 'credit'] | How are energy credits calculated each month?
  [PASS] top_chunks=[6, 5, 3, 4] expects~['75', 'termination'] | What is the early-termination fee within the first y
  [PASS] top_chunks=[2, 1, 7, 9] expects~['4,200', 'ridgeway'] | How many solar panels are in the Ridgeway array?
  [PASS] top_chunks=[5, 9, 8, 6] expects~['billing', 'support@greenleaf'] | Who do members contact for billing questions?
  [PASS] top_chunks=[1, 0, 2, 8] expects~['15', 'percent'] | What percentage of profits are reinvested into commu

Retrieval config: hybrid
  [PASS] top_chunks=[3, 6, 4, 5] expects~['120', 'standard'] | What is the annual membership fee for a Standard mem
  [PASS] top_chunks=[0, 6, 9, 8]

## 13. System Metrics Report

A summary of all the key parameters and measured performance in one place.

In [ ]:
def metrics_report():
    n_chunks = len(chunks)
    avg_chars = sum(c["n_chars"] for c in chunks) / max(1, n_chunks)
    L = []
    L.append("=" * 60)
    L.append(" RAG SYSTEM - METRICS REPORT")
    L.append("=" * 60)
    L.append("")
    L.append("[1] Document and chunking profile")
    L.append("    Source document      : " + doc_meta["source"])
    L.append("    Raw characters       : {:,}".format(len(raw_text)))
    L.append("    Chunk size (target)  : {} chars".format(CONFIG["CHUNK_SIZE"]))
    L.append("    Chunk overlap        : {} chars".format(CONFIG["CHUNK_OVERLAP"]))
    L.append("    Number of chunks     : {}".format(n_chunks))
    L.append("    Avg chunk length     : {:.0f} chars".format(avg_chars))
    L.append("")
    L.append("[2] Embedding model")
    L.append("    Model                : " + CONFIG["EMBEDDING_MODEL"])
    L.append("    Embedding dimension  : {}".format(EMBED_DIM))
    L.append("    Vectors embedded     : {}".format(chunk_embeddings.shape[0]))
    L.append("")
    L.append("[3] Vector store")
    L.append("    Library              : FAISS")
    L.append("    Index type           : IndexFlatIP (exact)")
    L.append("    Similarity metric    : cosine (L2-normalized inner product)")
    L.append("    Vectors indexed      : {}".format(store.index.ntotal))
    L.append("")
    L.append("[4] Retrieval configuration")
    L.append("    top_k (to LLM)       : {}".format(CONFIG["TOP_K"]))
    L.append("    candidate_k          : {}".format(CONFIG["CANDIDATE_K"]))
    L.append("    Hybrid (BM25+dense)  : {} (RRF fusion)".format(CONFIG["USE_HYBRID"]))
    L.append("    Re-ranker            : {} ({})".format(CONFIG["USE_RERANK"], CONFIG["RERANK_MODEL"]))
    L.append("")
    L.append("[5] Language model (generation)")
    L.append("    Primary              : {} (Anthropic SDK)".format(CONFIG["CLAUDE_MODEL"]))
    L.append("    Fallback (free)      : " + CONFIG["FREE_MODEL"])
    L.append("    Active backend       : " + ("Claude" if USE_CLAUDE else "free model"))
    L.append("")
    L.append("[6] Retrieval accuracy (validation, hit@{})".format(CONFIG["TOP_K"]))
    for name, hr in RETRIEVAL_SUMMARY.items():
        L.append("    {:<20}: {:.2f}".format(name, hr))
    L.append("=" * 60)
    print("\n".join(L))

metrics_report()

 RAG SYSTEM - METRICS REPORT

[1] Document and chunking profile
    Source document      : built-in sample document
    Raw characters       : 5,180
    Chunk size (target)  : 700 chars
    Chunk overlap        : 120 chars
    Number of chunks     : 10
    Avg chunk length     : 613 chars

[2] Embedding model
    Model                : sentence-transformers/all-MiniLM-L6-v2
    Embedding dimension  : 384
    Vectors embedded     : 10

[3] Vector store
    Library              : FAISS
    Index type           : IndexFlatIP (exact)
    Similarity metric    : cosine (L2-normalized inner product)
    Vectors indexed      : 10

[4] Retrieval configuration
    top_k (to LLM)       : 4
    candidate_k          : 12
    Hybrid (BM25+dense)  : True (RRF fusion)
    Re-ranker            : True (cross-encoder/ms-marco-MiniLM-L-6-v2)

[5] Language model (generation)
    Primary              : claude-sonnet-4-20250514 (Anthropic SDK)
    Fallback (free)      : google/flan-t5-base
    Active backend

## 14. Interactive Q&A

Uncomment the last line to start an interactive loop where you can type your own questions.

**To use your own document**, re-run ingestion and rebuild the index:

```python
raw_text, doc_meta = ingest("upload")                 # Colab file picker
# raw_text, doc_meta = ingest("pdf", path="/content/mydoc.pdf")
# raw_text, doc_meta = ingest("hf", name="vectara/open_ragbench")
chunks = chunk_text(raw_text)
chunk_embeddings = embed([c["text"] for c in chunks])
store = VectorStore(EMBED_DIM); store.add(chunk_embeddings, chunks)
bm25 = BM25Okapi([_tok(c["text"]) for c in chunks])
```

In [ ]:
def interactive_qa():
    print("Interactive RAG Q&A — type a question (blank line to exit).")
    while True:
        try:
            q = input("\nYour question: ").strip()
        except EOFError:
            break
        if not q:
            print("Done.")
            break
        rag_answer(q)

# interactive_qa()   # uncomment in Colab to start

## 15. Limitations and Next Steps

- **Flan-T5 is extractive only** — it just pulls a span from the context, it doesn't combine facts across passages. Claude does a much better job here but needs an API key.
- **Exact search doesn't scale** — `IndexFlatIP` works for small corpora. For millions of chunks you'd need an approximate index (IVF, HNSW).
- **No answer evaluation** — the validation harness checks retrieval quality (hit@k) but doesn't score the generated answers themselves. Something like RAGAS could be added for that.
- **Single document** — this indexes one document at a time. Extending to a multi-document corpus would just need a source ID on each chunk for attribution.